# Study 817 — Realized-Volatility Trend — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the level-vs-trend additivity regression, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4083, 'spread_bps': 0.94, 't_nw': 0.86, 't_1s': 0.83, 'lo_bps': 7.74, 'hi_bps': 6.8, 'welch_t': 0.36, 'gross_sharpe': 0.21, 'placebo_obs': 0.94, 'placebo_mean': -0.021, 'placebo_sd': 0.883, 'placebo_p': 0.121, 'placebo_sigma': 1.09, 'placebo_draws': 1000, 'level_bps': -4.74, 'level_t': -2.61, 'corr': 0.065, 'beta': 0.038, 'alpha_bps': 1.12, 'alpha_t': 1.02, 'era_early_bps': 1.52, 'era_early_t': 1.15, 'era_early_n': 1949, 'era_late_bps': 0.41, 'era_late_t': 0.24, 'era_late_n': 2134, 'timer_1_gross': 0.94, 'timer_1_cost': 2.14, 'timer_1_net': -1.2, 'timer_1_t': -1.06, 'timer_5_gross': 0.94, 'timer_5_cost': 10.14, 'timer_5_net': -9.2, 'timer_5_t': -8.14, 'null_mean_t': -0.18, 'null_sd_t': 1.03, 'null_fire': 0, 'planted_t': 9.49, 'planted_welch': 9.29}

## The headline — long-falling-vol / short-rising-vol spread

Daily equal-weight bottom-30% (falling) minus top-30% (rising) vol-trend spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : falling-vol {R['lo_bps']:+.2f} vs rising-vol {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +0.94 bps/day  NW(10) t = +0.86  one-sample t = +0.83
books         : falling-vol +7.74 vs rising-vol +6.80 bps (Welch t = +0.36)
gross Sharpe  : 0.21 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.3f} "
      f"({R['placebo_sigma']:+.2f} sigma into the right tail)")

observed +0.94 bps vs placebo mean -0.021 (sd 0.883) -> p = 0.121 (+1.09 sigma into the right tail)


## Additivity — is the vol TREND anything beyond the low-vol LEVEL (330)?

Build the plain low-vol level sort (long low 63d-vol, short high 63d-vol), then regress the trend spread on the level spread and read the residual NW *t*.

In [4]:
print(f"low-vol LEVEL spread : {R['level_bps']:+.2f} bps/day (NW t = {R['level_t']:+.2f})  "
      f"# itself inverted on mega-caps")
print(f"corr(trend, level)   : {R['corr']:+.3f}   beta = {R['beta']:+.3f}  -> near-orthogonal")
print(f"trend alpha vs level : {R['alpha_bps']:+.2f} bps/day (NW t = {R['alpha_t']:+.2f})  "
      f"# distinct axis, equally empty")

low-vol LEVEL spread : -4.74 bps/day (NW t = -2.61)  # itself inverted on mega-caps
corr(trend, level)   : +0.065   beta = +0.038  -> near-orthogonal
trend alpha vs level : +1.12 bps/day (NW t = +1.02)  # distinct axis, equally empty


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1949): +1.52 bps  NW t = +1.15
2018-2026 (n=2134): +0.41 bps  NW t = +0.24


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +0.94 -> net -1.20 bps/day (cost 2.14/day, t=-1.06)
5 bps one-way: gross +0.94 -> net -9.20 bps/day (cost 10.14/day, t=-8.14)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from vol_trend import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=817+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=817, n_assets=40, n_days=1500))
print(f"planted (edge=0.0015): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.14 (sd 1.00), |t|>=2 in 0/8


planted (edge=0.0015): NW t = +9.49, Welch t = +9.29


## Verdict

- **Signal — None.** The vol *trend* carries no reliable cross-sectional signal on 50 liquid US mega-caps: the long-falling-vol / short-rising-vol spread is **+0.94 bps/day** (NW *t* = **+0.86**) — the claimed sign but statistically zero, only +1.09σ into the placebo (p = 0.12), weak in both eras (*t* = +1.15 / +0.24). It is **not additive**: near-orthogonal to the level sort (corr +0.065) yet its alpha net of the level is just +1.12 bps/day (*t* = +1.02). The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +9.49, fires on 0/20 nulls), so the flat result is real, not machinery.
- **Tradability — Mirage.** The +0.94 bps/day gross edge is smaller than the round-trip friction (2.14 bps/day) at 1 bp one-way, so the book is net **-1.20 bps/day** (*t* = -1.06); at 5 bps **-9.20 bps/day**.